In [1]:
import os
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [2]:
from src.loader import load_candidates
from src.preprocessing import prepare_data
from src.feature_engineering import build_all_features

In [3]:
candidates = load_candidates()

data = prepare_data(candidates)

features = build_all_features(
    profiles_df=data["profiles"],
    career_df=data["career"],
    skill_df=data["skills"],
    signals_df=data["signals"]
)


In [4]:
semantic_scores = pd.read_parquet(
    "../data/semantic_scores.parquet"
)

behavior_scores = pd.read_parquet(
    "../data/behavior_scores.parquet"
)

In [5]:
ranking_df = (
    features
    .merge(semantic_scores, on="candidate_id")
    .merge(behavior_scores, on="candidate_id")
)

ranking_df.shape

(100000, 64)

In [6]:
from src.ranking import *

In [7]:
technical = build_technical_score(features)

product = build_product_fit_score(features)

experience = build_experience_fit_score(features)

ranking_df = build_final_score(
    ranking_df,
    technical,
    product,
    experience
)

In [8]:
ranking_df = rank_candidates(

    ranking_df

)

In [9]:
ranking_df.head(20)

,candidate_id,years_of_experience,current_title,total_jobs,unique_companies,avg_job_duration,shortest_job,longest_job,product_company_ratio,service_company_ratio,...,semantic_similarity,availability_score,recruiter_interest_score,trust_score,engagement_score,behavior_score,technical_score,product_fit_score,experience_fit_score,final_score
0,CAND_0077337,7.0,Staff Machine Learning Engineer,4,4,20.750000,6,44,0.500000,0.0,...,0.782215,0.907800,0.292314,0.8584,0.6525,0.851117,0.839801,0.804665,1.000000,0.833849
1,CAND_0046064,8.9,Senior NLP Engineer,3,3,35.333333,34,36,0.000000,0.0,...,0.877985,0.876148,0.640024,0.9012,0.4446,0.912841,0.833262,0.521866,0.857143,0.832100
2,CAND_0081846,6.7,Lead AI Engineer,2,2,39.500000,27,52,1.000000,0.0,...,0.849290,0.861580,0.508979,0.9828,0.4864,0.893847,0.654356,1.000000,1.000000,0.827635
3,CAND_0052682,6.6,NLP Engineer,2,2,39.000000,32,46,0.000000,0.0,...,0.752276,0.910898,0.542406,0.9416,0.7678,1.000000,0.864248,0.451895,1.000000,0.817761
4,CAND_0002025,5.9,Senior AI Engineer,2,2,35.000000,28,42,0.000000,0.0,...,0.949568,0.896470,0.381545,0.9272,0.6153,0.886659,0.706972,0.539359,0.857143,0.817090
5,CAND_0008425,7.8,Senior NLP Engineer,3,3,30.666667,21,46,0.000000,0.0,...,0.805199,0.801739,0.674128,0.8288,0.5291,0.891084,0.783039,0.591837,1.000000,0.809577
6,CAND_0083307,7.8,Search Engineer,4,4,23.000000,7,51,0.250000,0.0,...,0.696485,0.773015,0.295603,0.8120,0.5819,0.750593,0.917850,0.667638,1.000000,0.798478
7,CAND_0088025,8.6,Staff Machine Learning Engineer,3,3,34.000000,13,45,0.000000,0.0,...,0.775883,0.843167,0.554380,0.8412,0.7877,0.946947,0.797946,0.521866,0.857143,0.790886
8,CAND_0007009,7.9,Recommendation Systems Engineer,3,3,31.333333,14,48,0.000000,0.0,...,0.635762,0.827429,0.266034,0.7312,0.6199,0.755049,1.000000,0.521866,1.000000,0.787961
9,CAND_0081686,6.0,Search Engineer,3,3,23.666667,13,30,0.333333,0.0,...,0.758746,0.514925,0.458154,0.8288,0.5138,0.664189,0.825023,0.701652,1.000000,0.782862


In [10]:
ranking_summary = build_submission_features(ranking_df)

ranking_summary.head(20)

,candidate_id,current_title,years_of_experience,semantic_similarity,technical_score,behavior_score,product_fit_score,experience_fit_score,final_score
0,CAND_0077337,Staff Machine Learning Engineer,7.0,0.782215,0.839801,0.851117,0.804665,1.000000,0.833849
1,CAND_0046064,Senior NLP Engineer,8.9,0.877985,0.833262,0.912841,0.521866,0.857143,0.832100
2,CAND_0081846,Lead AI Engineer,6.7,0.849290,0.654356,0.893847,1.000000,1.000000,0.827635
3,CAND_0052682,NLP Engineer,6.6,0.752276,0.864248,1.000000,0.451895,1.000000,0.817761
4,CAND_0002025,Senior AI Engineer,5.9,0.949568,0.706972,0.886659,0.539359,0.857143,0.817090
5,CAND_0008425,Senior NLP Engineer,7.8,0.805199,0.783039,0.891084,0.591837,1.000000,0.809577
6,CAND_0083307,Search Engineer,7.8,0.696485,0.917850,0.750593,0.667638,1.000000,0.798478
7,CAND_0088025,Staff Machine Learning Engineer,8.6,0.775883,0.797946,0.946947,0.521866,0.857143,0.790886
8,CAND_0007009,Recommendation Systems Engineer,7.9,0.635762,1.000000,0.755049,0.521866,1.000000,0.787961
9,CAND_0081686,Search Engineer,6.0,0.758746,0.825023,0.664189,0.701652,1.000000,0.782862
